In [1]:
import pandas as pd
from pyspark import SparkConf
from pyspark.sql import SparkSession, functions as f, Window

In [2]:
parameters = {
    "spark.driver.maxResultSize": "3g",
    "spark.hadoop.fs.s3a.impl": "org.apache.hadoop.fs.s3a.S3AFileSystem",
    "spark.sql.execution.arrow.pyspark.enabled": True,

    # https://docs.kedro.org/en/stable/integrations/pyspark_integration.html#tips-for-maximising-concurrency-using-threadrunner
    "spark.scheduler.mode": "FAIR",
    "spark.driver.extraJavaOptions": "-Djava.security.manager=allow",
    "spark.executor.extraJavaOptions": "-Djava.security.manager=allow",

    "spark.sql.legacy.parquet.nanosAsLong": True,
    "spark.sql.legacy.timeParserPolicy": "LEGACY",
    "spark.driver.memory": "40g",
}

spark_conf = SparkConf().setAll(parameters.items())

In [3]:
spark = SparkSession.builder.appName('Loyalty').enableHiveSupport().config(conf=spark_conf).getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/21 15:24:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
ftr_churn = spark.read.parquet("../data/04_feature/ftr_churn")

In [8]:
working_ids_list = ftr_churn.filter(
    f.col("_observ_end_dt") == "2024-01-31"
).filter(
    f.col("is_active") > 0
).filter(
    f.col("is_new_joiner") > 0
).select(
    f.collect_set('_id').alias('_id')
).first()['_id']

print(len(working_ids_list))

35735


In [10]:
new_joiner_df = ftr_churn.filter(
    f.col("_observ_end_dt").isin(["2024-01-31", "2024-04-30", "2024-07-31", "2024-10-31", "2025-01-31"])
).filter(
    f.col("_id").isin(working_ids_list)
).select(
    "_id",
    "_observ_end_dt",
    "last_transaction_dt",
    "is_new_joiner", "is_uncommited", "is_potential_loyal", "is_loyal", "is_lost", "is_gone",
    "total_number_of_visits", "months_since_last_visit", "months_since_first_visit",
    "customer_mineral_oil", "customer_synthetic_oil", "total_number_of_visits_last_12_months", "expected_number_of_visits_mineral_oil", "expected_number_of_visits_synthetic_oil",
    "month_avg_mileage_per_day_nullif_past_11_next_0_months",
).orderBy("_id", "_observ_end_dt")

new_joiner_df.show(15, truncate=False)

25/05/21 15:39:00 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB


+---------------------+--------------+-------------------+-------------+-------------+------------------+--------+-------+-------+----------------------+-----------------------+------------------------+--------------------+----------------------+-------------------------------------+-------------------------------------+---------------------------------------+------------------------------------------------------+
|_id                  |_observ_end_dt|last_transaction_dt|is_new_joiner|is_uncommited|is_potential_loyal|is_loyal|is_lost|is_gone|total_number_of_visits|months_since_last_visit|months_since_first_visit|customer_mineral_oil|customer_synthetic_oil|total_number_of_visits_last_12_months|expected_number_of_visits_mineral_oil|expected_number_of_visits_synthetic_oil|month_avg_mileage_per_day_nullif_past_11_next_0_months|
+---------------------+--------------+-------------------+-------------+-------------+------------------+--------+-------+-------+----------------------+-----------

In [ ]:
new_joiner_df.write.csv("new_joiner_table", sep=";", header=True,)

25/05/21 15:52:44 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
25/05/21 15:53:03 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
25/05/21 15:53:22 WARN DAGScheduler: Broadcasting large task binary with size 3.7 MiB


In [16]:
import tqdm
import glob

In [18]:
new_joiner_pdf_list = []
for file in tqdm.tqdm(glob.glob(f'new_joiner_table/*.csv', recursive=True)):
    pdf = pd.read_csv(file, sep=";")
    new_joiner_pdf_list.append(pdf)

new_joiner_pdf = pd.concat(new_joiner_pdf_list, axis=0)

100%|██████████| 5/5 [00:00<00:00, 28.61it/s]


In [27]:
import numpy as np

In [20]:
new_joiner_pdf.shape

(178675, 18)

In [28]:
new_joiner_pdf["is_potential_loyal"] = np.where(
    new_joiner_pdf["is_loyal"] == 1,
    0,
    new_joiner_pdf["is_potential_loyal"]
)

new_joiner_pdf["is_uncommited"] = np.where(
    (new_joiner_pdf["is_loyal"] == 1),
    0,
    new_joiner_pdf["is_uncommited"]
)

new_joiner_pdf["is_uncommited"] = np.where(
    (new_joiner_pdf["is_potential_loyal"] == 1),
    0,
    new_joiner_pdf["is_uncommited"]
)

In [32]:
new_joiner_pdf["mck_segment"] = None
new_joiner_pdf["mck_segment"] = np.where(new_joiner_pdf["is_new_joiner"]==1, "New Joiner",new_joiner_pdf["mck_segment"])
new_joiner_pdf["mck_segment"] = np.where(new_joiner_pdf["is_uncommited"]==1, "Uncommited",new_joiner_pdf["mck_segment"])
new_joiner_pdf["mck_segment"] = np.where(new_joiner_pdf["is_potential_loyal"]==1, "Potential Loyal",new_joiner_pdf["mck_segment"])
new_joiner_pdf["mck_segment"] = np.where(new_joiner_pdf["is_loyal"]==1, "Loyal",new_joiner_pdf["mck_segment"])
new_joiner_pdf["mck_segment"] = np.where(new_joiner_pdf["is_lost"]==1, "Lost",new_joiner_pdf["mck_segment"])
new_joiner_pdf["mck_segment"] = np.where(new_joiner_pdf["is_gone"]==1, "Gone",new_joiner_pdf["mck_segment"])

In [33]:
new_joiner_pdf.to_excel("new_joiner_table.xlsx", index=False, header=True, engine="openpyxl")

25/05/21 19:49:01 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 176850 ms exceeds timeout 120000 ms
25/05/21 19:49:01 WARN SparkContext: Killing executors is not supported by current scheduler.
25/05/21 19:49:02 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:80)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:642)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1223)
	at o

In [29]:
new_joiner_pdf_grouped = new_joiner_pdf.groupby(["_observ_end_dt"]).agg({
    "is_new_joiner": "sum",
    "is_uncommited": "sum",
    "is_potential_loyal": "sum",
    "is_loyal": "sum",
    "is_lost": "sum",
    "is_gone": "sum",
})

new_joiner_pdf_grouped

,is_new_joiner,is_uncommited,is_potential_loyal,is_loyal,is_lost,is_gone
_observ_end_dt,,,,,,
2024-01-31,35735,0,0,0,0,0
2024-04-30,26380,8921,264,170,0,0
2024-07-31,21055,13004,686,990,0,0
2024-10-31,18655,13635,1183,2262,0,0
2025-01-31,0,14923,1132,2262,17418,0


In [30]:
new_joiner_pdf_grouped.div(35735).round(2).mul(100)

,is_new_joiner,is_uncommited,is_potential_loyal,is_loyal,is_lost,is_gone
_observ_end_dt,,,,,,
2024-01-31,100.0,0.0,0.0,0.0,0.0,0.0
2024-04-30,74.0,25.0,1.0,0.0,0.0,0.0
2024-07-31,59.0,36.0,2.0,3.0,0.0,0.0
2024-10-31,52.0,38.0,3.0,6.0,0.0,0.0
2025-01-31,0.0,42.0,3.0,6.0,49.0,0.0
